# Recruitment Chatbot — Evaluation

**Two evaluations:**
1. **Multi-Agent System** — Accuracy & Confusion Matrix classifying each recruiter turn as `end` / `continue` / `schedule`
2. **Exit Advisor — Prompt Engineering** — comparing a basic prompt (V1) vs the engineered prompt (V2, live in production) on the `end` decision specifically

**Metrics:** Accuracy, Confusion Matrix, Classification Report

## 1. Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from dotenv import load_dotenv
from openai import OpenAI

from app.modules.system_evaluation.system_evaluation import load_test_cases, classify_action
from app.modules.exit_advisor_prompt_engineering.exit_advisor_prompt_engineering import compare_prompts

load_dotenv()
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
client = OpenAI(api_key=OPENAI_API_KEY)
print('Setup complete.')

## 2. Load Labeled Data

In [ ]:
test_cases = load_test_cases()
df = pd.DataFrame(test_cases)
df.rename(columns={'expected': 'true_label'}, inplace=True)
print(f'Total labeled samples: {len(df)}')
print(df['true_label'].value_counts())
df.head(3)

## 3. Prediction Helper

In [ ]:
def plot_confusion_matrix(y_true, y_pred, labels, title):
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    df_cm = pd.DataFrame(cm, index=labels, columns=labels)
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(df_cm, annot=True, fmt='d', cmap='Blues', linewidths=0.5, ax=ax)
    ax.set_title(title, fontsize=14, pad=12)
    ax.set_xlabel('Predicted', fontsize=11)
    ax.set_ylabel('Actual', fontsize=11)
    plt.tight_layout()
    plt.show()

print('Helpers ready.')

## 4. Multi-Agent System Evaluation (End / Continue / Schedule)

In [ ]:
print('Running multi-agent system classification on all labeled turns...')
df['predicted'] = df['context'].apply(lambda ctx: classify_action(ctx, client))
system_accuracy = accuracy_score(df['true_label'], df['predicted'])
print(f'System Accuracy: {system_accuracy:.2%}')

In [ ]:
labels = ['end', 'continue', 'schedule']
print(classification_report(df['true_label'], df['predicted'], labels=labels))
plot_confusion_matrix(df['true_label'], df['predicted'], labels=labels, title=f'Multi-Agent System — Accuracy: {system_accuracy:.2%}')

## 5. Exit Advisor — Prompt Engineering (V1 vs V2)

In [ ]:
print('Comparing basic prompt (V1) vs engineered prompt (V2) for the Exit Advisor...')
comparison = compare_prompts()
v1_acc = comparison['v1']['accuracy']
v2_acc = comparison['v2']['accuracy']
print(f'V1 (basic) accuracy:     {v1_acc:.2%}')
print(f'V2 (prompt engineering): {v2_acc:.2%}')

In [ ]:
print('V1 confusion matrix (should_exit True/False):')
print(f"  TP={comparison['v1']['tp']}  FP={comparison['v1']['fp']}  TN={comparison['v1']['tn']}  FN={comparison['v1']['fn']}")
print('V2 confusion matrix (should_exit True/False):')
print(f"  TP={comparison['v2']['tp']}  FP={comparison['v2']['fp']}  TN={comparison['v2']['tn']}  FN={comparison['v2']['fn']}")

## 6. V1 vs V2 Accuracy Comparison

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(['V1 (basic)', 'V2 (prompt engineering)'], [v1_acc*100, v2_acc*100], color=['#4C72B0','#55A868'], width=0.4)
for bar, acc in zip(bars, [v1_acc, v2_acc]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height()+0.5, f'{acc:.2%}', ha='center', va='bottom', fontsize=12, fontweight='bold')
ax.set_ylim(0, 110)
ax.set_ylabel('Accuracy (%)')
ax.set_title('Exit Advisor — Prompt Engineering Improvement')
plt.tight_layout()
plt.show()
print(f'Improvement: {(v2_acc - v1_acc)*100:+.1f} percentage points')